# Mid-Block Expansion 细粒度观察

逐样本查看 **mask → token** 随 step 变化的完整过程。

**观察内容：**
- 每步哪些 position 从 [MASK] 变成了 token（heatmap）
- expansion 触发时的 block 边界变化
- 三组 config 在同一样本上的 decode 行为对比

**样本：** GSM8K x 5 + MBPP x 5

## 1. 环境设置

In [ ]:
import os
import torch
import gc

# Set GPU (modify as needed)
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'

# Environment settings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

# Change to llada directory
os.chdir('llada')

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 加载模型

In [ ]:
from transformers import AutoTokenizer
from model.modeling_llada import LLaDAModelLM

device = 'cuda'
model_name = 'GSAI-ML/LLaDA-8B-Instruct'

print(f"Loading model: {model_name}")
model = LLaDAModelLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Model loaded!")

## 3. 准备样本（GSM8K x 5 + MBPP x 5）

In [ ]:
gsm_questions = [
    "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
    "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?",
    "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?",
    "Lily can run 12 kilometers per hour for 4 hours. After that, she runs 6 kilometers per hour. How many kilometers can she run in 8 hours?",
    "A store sells notebooks for $3 each. If you buy 5 or more, you get a 20% discount. How much do 7 notebooks cost?",
]

mbpp_questions = [
    "Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix.",
    "Write a function to find the similar elements from the given two tuple lists.",
    "Write a python function to identify non-prime numbers.",
    "Write a function to find the longest common subsequence of two strings.",
    "Write a function to sort a list of tuples by the second element.",
]

all_samples = []
for i, q in enumerate(gsm_questions):
    m = [{"role": "user", "content": q}]
    prompt_text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    input_ids = torch.tensor(tokenizer(prompt_text)['input_ids']).to(device).unsqueeze(0)
    all_samples.append({'idx': i, 'task': 'gsm8k', 'question': q[:80], 'input_ids': input_ids})

for i, q in enumerate(mbpp_questions):
    m = [{"role": "user", "content": q}]
    prompt_text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    input_ids = torch.tensor(tokenizer(prompt_text)['input_ids']).to(device).unsqueeze(0)
    all_samples.append({'idx': i, 'task': 'mbpp', 'question': q[:80], 'input_ids': input_ids})

print(f"Total samples: {len(all_samples)}")
for s in all_samples:
    print(f"  [{s['task']}#{s['idx']}] {s['question'][:60]}...")

## 4. 运行三组配置并收集 mask snapshot

In [ ]:
from generate import generate_with_dual_cache_expand
from tqdm.auto import tqdm
import numpy as np

GEN_LENGTH   = 256
STEPS        = 256
BLOCK_LENGTH = 32
THRESHOLD    = 0.9
MASK_ID      = 126336

configs = {
    'baseline':         {'mid_trigger_ratio': 0.0, 'rewarm_on_expand': True},
    'expand_no_rewarm': {'mid_trigger_ratio': 0.5, 'rewarm_on_expand': False},
    'expand_rewarm':    {'mid_trigger_ratio': 0.5, 'rewarm_on_expand': True},
}

# results[config_name][sample_idx] = {'step_records': [...], 'gen_text': str, 'nfe': int}
results = {c: [] for c in configs}

for cname, cfg in configs.items():
    print(f"\n{'='*60}")
    print(f"Config: {cname}")
    print(f"{'='*60}")
    for sample in tqdm(all_samples, desc=cname):
        input_ids = sample['input_ids']
        prompt_len = input_ids.shape[1]

        with torch.inference_mode():
            x, nfe, step_records = generate_with_dual_cache_expand(
                model, input_ids,
                steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
                temperature=0., threshold=THRESHOLD,
                mid_trigger_ratio=cfg['mid_trigger_ratio'],
                rewarm_on_expand=cfg['rewarm_on_expand'],
                record_steps=True,
            )

        gen_token_ids = x[0, prompt_len:].cpu().tolist()  # 保存完整 token ids
        gen_text = tokenizer.decode(gen_token_ids, skip_special_tokens=True)
        results[cname].append({
            'task': sample['task'],
            'idx': sample['idx'],
            'question': sample['question'],
            'gen_text': gen_text,
            'gen_token_ids': gen_token_ids,
            'nfe': nfe,
            'step_records': step_records,
        })

print("\nAll done.")

## 5. Mask 演化热力图

每张图：X 轴 = 生成区域的 position（0~255），Y 轴 = step 序号。
- **白色** = 已解码 token
- **深色** = 仍然是 [MASK]
- **红色竖线** = block 边界（每 32 个 token）
- **黄色标记** = expansion 触发的 step

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

config_labels = {
    'baseline': 'Baseline',
    'expand_no_rewarm': 'Expand (no rewarm)',
    'expand_rewarm': 'Expand (rewarm)',
}

def plot_mask_heatmap(step_records, title, ax):
    """绘制单个样本的 mask 演化热力图。"""
    snapshots = []
    step_types = []
    for r in step_records:
        snap = r.get('mask_snapshot')
        if snap is not None:
            snapshots.append(snap)
            step_types.append(r['type'])

    if not snapshots:
        ax.text(0.5, 0.5, 'No mask snapshots', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return

    mat = np.array(snapshots, dtype=np.float32)  # (n_steps, gen_length), 1=MASK 0=decoded
    n_steps, gen_len = mat.shape

    # 用自定义 colormap: 0=白(decoded), 1=深蓝(MASK)
    cmap = ListedColormap(['#FFFFFF', '#1565C0'])
    ax.imshow(mat, aspect='auto', cmap=cmap, interpolation='nearest', vmin=0, vmax=1)

    # Block 边界竖线
    for b in range(BLOCK_LENGTH, gen_len, BLOCK_LENGTH):
        ax.axvline(x=b - 0.5, color='red', linewidth=0.5, alpha=0.5)

    # 标记 expand step（黄色三角）
    for step_i, stype in enumerate(step_types):
        if stype == 'expand':
            ax.plot(-1, step_i, marker='>', color='#FF9800', markersize=6, clip_on=False)

    ax.set_xlabel('Position')
    ax.set_ylabel('Step')
    ax.set_title(title, fontsize=10)


# --- 为每个样本画三组对比 ---
os.makedirs('../eval_results', exist_ok=True)

for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    task_name = sample_info['task'].upper()
    q_short = sample_info['question'][:50]

    fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)

    for ax_i, cname in enumerate(configs.keys()):
        res = results[cname][sample_i]
        title = f"{config_labels[cname]}\nNFE={res['nfe']}"
        plot_mask_heatmap(res['step_records'], title, axes[ax_i])

    fig.suptitle(f"[{task_name} #{sample_info['idx']}] {q_short}...", fontsize=12, fontweight='bold')

    # Legend
    legend_elements = [
        mpatches.Patch(facecolor='#1565C0', label='[MASK]'),
        mpatches.Patch(facecolor='#FFFFFF', edgecolor='gray', label='Decoded'),
        plt.Line2D([0], [0], color='red', linewidth=1, label='Block boundary'),
        plt.Line2D([0], [0], marker='>', color='#FF9800', linestyle='None', markersize=8, label='Expand step'),
    ]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=9, ncol=4)

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    fname = f"../eval_results/detail_{sample_info['task']}_{sample_info['idx']}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

## 6. 逐步 Diff 视图

每步新解码了哪些 position（上一步是 MASK → 本步变成 token 的位置）。
- **绿色** = 本步新解码
- **灰色** = 之前已解码
- **深蓝** = 仍是 MASK

In [ ]:
def plot_diff_heatmap(step_records, title, ax):
    """绘制逐步 diff 热力图：本步新解码的 position 用绿色高亮。"""
    snapshots = []
    for r in step_records:
        snap = r.get('mask_snapshot')
        if snap is not None:
            snapshots.append(snap)

    if not snapshots:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return

    mat = np.array(snapshots, dtype=np.int8)  # 1=MASK, 0=decoded
    n_steps, gen_len = mat.shape

    # 构造 diff 矩阵: 0=still_mask, 1=already_decoded, 2=newly_decoded
    diff_mat = np.zeros_like(mat, dtype=np.int8)
    for s in range(n_steps):
        if s == 0:
            diff_mat[s] = np.where(mat[s] == 1, 0, 2)  # 第0步: 非mask的都是"新解码"
        else:
            newly = (mat[s-1] == 1) & (mat[s] == 0)  # 上步是mask，本步不是
            already = (mat[s] == 0) & (~newly)         # 本步不是mask，也不是新解码
            diff_mat[s] = np.where(mat[s] == 1, 0, np.where(newly, 2, 1))

    # 0=MASK(深蓝), 1=already_decoded(浅灰), 2=newly_decoded(绿色)
    cmap = ListedColormap(['#1565C0', '#E0E0E0', '#4CAF50'])
    ax.imshow(diff_mat, aspect='auto', cmap=cmap, interpolation='nearest', vmin=0, vmax=2)

    for b in range(BLOCK_LENGTH, gen_len, BLOCK_LENGTH):
        ax.axvline(x=b - 0.5, color='red', linewidth=0.5, alpha=0.5)

    ax.set_xlabel('Position')
    ax.set_ylabel('Step')
    ax.set_title(title, fontsize=10)


for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    task_name = sample_info['task'].upper()
    q_short = sample_info['question'][:50]

    fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)

    for ax_i, cname in enumerate(configs.keys()):
        res = results[cname][sample_i]
        title = f"{config_labels[cname]}\nNFE={res['nfe']}"
        plot_diff_heatmap(res['step_records'], title, axes[ax_i])

    fig.suptitle(f"[{task_name} #{sample_info['idx']}] Step Diff: {q_short}...", fontsize=12, fontweight='bold')

    legend_elements = [
        mpatches.Patch(facecolor='#1565C0', label='[MASK]'),
        mpatches.Patch(facecolor='#E0E0E0', label='Already decoded'),
        mpatches.Patch(facecolor='#4CAF50', label='Newly decoded this step'),
        plt.Line2D([0], [0], color='red', linewidth=1, label='Block boundary'),
    ]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=9, ncol=4)

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    fname = f"../eval_results/diff_{sample_info['task']}_{sample_info['idx']}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

## 7. 每步解码数 + 累计进度（逐样本对比）

In [ ]:
colors_cfg = {'baseline': '#2196F3', 'expand_no_rewarm': '#FF9800', 'expand_rewarm': '#4CAF50'}

for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    task_name = sample_info['task'].upper()
    q_short = sample_info['question'][:50]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

    for cname in configs.keys():
        res = results[cname][sample_i]
        sr = res['step_records']
        steps_x = [r['global_step'] for r in sr]
        transferred = [r['transferred'] for r in sr]
        cumulative = np.cumsum(transferred) / GEN_LENGTH

        color = colors_cfg[cname]
        label = f"{config_labels[cname]} (NFE={res['nfe']})"

        # 左图: 每步解码数
        ax1.bar([s + list(configs.keys()).index(cname) * 0.25 for s in steps_x],
                transferred, width=0.25, color=color, alpha=0.7, label=label)

        # 右图: 累计进度
        ax2.plot(steps_x, cumulative, color=color, linewidth=2, label=label, marker='.', markersize=3)

        # 标记 expand steps
        for r in sr:
            if r['type'] == 'expand':
                ax2.axvline(x=r['global_step'], color=color, linestyle='--', alpha=0.4)

    ax1.set_xlabel('Global Step')
    ax1.set_ylabel('Tokens Transferred')
    ax1.set_title('Per-Step Decode Count')
    ax1.legend(fontsize=8)

    ax2.set_xlabel('Global Step')
    ax2.set_ylabel('Cumulative Decoded Ratio')
    ax2.set_title('Decode Progress')
    ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4)
    ax2.set_ylim(0, 1.1)
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    fig.suptitle(f"[{task_name} #{sample_info['idx']}] {q_short}...", fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

## 8. 交互式逐步解码查看器（HTML）

用滑块选 step，完整展示每一步的 mask/token 状态。
- <span style="color:red">[MASK]</span> = 仍未解码
- <span style="color:green;font-weight:bold">token</span> = 本步新解码
- 正常文字 = 之前已解码

In [ ]:
from IPython.display import display, HTML
import html as html_lib
import ipywidgets as widgets

def build_interactive_viewer(sample_i, config_name):
    """为一个样本+config 构建交互式 HTML 查看器。"""
    res = results[config_name][sample_i]
    sample_info = all_samples[sample_i]
    sr = res['step_records']
    final_ids = res['gen_token_ids']

    # 预渲染每一步的 HTML
    step_htmls = []
    prev_snap = None
    for r in sr:
        snap = r.get('mask_snapshot')
        if snap is None:
            continue

        pieces = []
        for pos, is_mask in enumerate(snap):
            if pos >= len(final_ids):
                break
            if is_mask:
                pieces.append('<span style="color:#e53935;font-size:11px;">[mask]</span>')
            else:
                tok_text = html_lib.escape(tokenizer.decode([final_ids[pos]]))
                # 判断是否是本步新解码的
                is_new = (prev_snap is not None and prev_snap[pos]) or (prev_snap is None and not is_mask)
                if is_new:
                    pieces.append(f'<span style="background:#c8e6c9;font-weight:bold;">{tok_text}</span>')
                else:
                    pieces.append(f'<span>{tok_text}</span>')

        step_type = r['type'].upper()
        type_icon = {'WARM': '🔥', 'REFINE': '🔧', 'EXPAND': '⚡'}.get(step_type, '')
        n_mask = sum(snap)
        header = (f'<b>Step {r["global_step"]}</b> {type_icon} {step_type} '
                  f'| +{r["transferred"]} tok | {n_mask} masks left | '
                  f'range [{r["range"][0]}:{r["range"][1]})')
        body = ''.join(pieces)
        step_htmls.append(f'<div style="margin-bottom:4px;font-size:12px;color:#666;">{header}</div>'
                          f'<div style="font-family:monospace;font-size:13px;line-height:1.6;'
                          f'word-wrap:break-word;white-space:pre-wrap;border:1px solid #ddd;'
                          f'padding:8px;border-radius:4px;background:#fafafa;">{body}</div>')
        prev_snap = snap

    return step_htmls


def show_interactive(sample_i, config_name):
    """显示带滑块的交互式查看器。"""
    res = results[config_name][sample_i]
    sample_info = all_samples[sample_i]
    step_htmls = build_interactive_viewer(sample_i, config_name)

    if not step_htmls:
        print("No step data.")
        return

    title_html = (f'<div style="font-size:14px;font-weight:bold;margin-bottom:8px;">'
                  f'[{sample_info["task"].upper()} #{sample_info["idx"]}] '
                  f'{config_labels[config_name]} | NFE={res["nfe"]}</div>'
                  f'<div style="font-size:12px;color:#555;margin-bottom:12px;">'
                  f'Q: {html_lib.escape(sample_info["question"])}</div>')

    output = widgets.HTML(value=title_html + step_htmls[0])
    slider = widgets.IntSlider(value=0, min=0, max=len(step_htmls)-1,
                                description='Step:', continuous_update=True,
                                layout=widgets.Layout(width='80%'))

    def on_change(change):
        output.value = title_html + step_htmls[change['new']]

    slider.observe(on_change, names='value')
    display(widgets.VBox([slider, output]))

## 9. 交互式查看：所有样本 x 所有 config

每个样本 x 每个 config 一个滑块，拖动 step 查看完整解码过程。

In [ ]:
for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    display(HTML(f'<h3 style="margin-top:24px;border-bottom:2px solid #333;padding-bottom:4px;">'
                 f'[{sample_info["task"].upper()} #{sample_info["idx"]}] '
                 f'{html_lib.escape(sample_info["question"])}</h3>'))
    for cname in configs.keys():
        show_interactive(sample_i, cname)

## 10. 全步骤展开查看（选一个样本+config 完整展示每一步）

In [ ]:
# ========== 修改这两个值选择要看的样本和配置 ==========
SAMPLE_IDX = 0         # 0~9 (0-4 gsm8k, 5-9 mbpp)
CONFIG_NAME = 'expand_rewarm'  # 'baseline' / 'expand_no_rewarm' / 'expand_rewarm'
# ===========================================================

step_htmls = build_interactive_viewer(SAMPLE_IDX, CONFIG_NAME)
res = results[CONFIG_NAME][SAMPLE_IDX]
sample_info = all_samples[SAMPLE_IDX]

full_html = (f'<h3>[{sample_info["task"].upper()} #{sample_info["idx"]}] '
             f'{config_labels[CONFIG_NAME]} | NFE={res["nfe"]} | '
             f'{len(step_htmls)} steps</h3>'
             f'<div style="font-size:12px;color:#555;margin-bottom:12px;">'
             f'Q: {html_lib.escape(sample_info["question"])}</div>')

for i, sh in enumerate(step_htmls):
    full_html += f'<div style="margin-bottom:16px;">{sh}</div>'

full_html += (f'<div style="margin-top:16px;padding:8px;background:#e8f5e9;border-radius:4px;">'
              f'<b>Final output:</b><br>'
              f'<span style="font-family:monospace;font-size:13px;">'
              f'{html_lib.escape(res["gen_text"])}</span></div>')

display(HTML(full_html))